# Estimating Optimal Temperatures

INSTALL MODEL

../seq2topt/model_topt_window.3_r2.0.57.pth

# LOAD PACKAGES

In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from seq2topt.functions import *
import timeit
import torch
from seq2topt.model import MultiAttModel
import esm
from src.biotools.fasta_tools import *
from seq2topt import *

/opt/homebrew/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


# LOAD tOPT MODEL

In [2]:
# path to model weights
topt_pth = '../seq2topt/model_topt_window.3_r2.0.57.pth';# you can download the weights from release v1.0.0
# define device
device = torch.device('cpu')
# define seq2Topt model
emb_dim= 320; window=3; n_head = 4; n_RD = 4;
model = MultiAttModel( emb_dim, window, n_head, n_RD)
model.to(device);
# load weights into the model
model.load_state_dict(torch.load( topt_pth, map_location=device  ));
model.eval();

# LOAD ESM-2

In [3]:
# We use esm2_t6_8M_UR50D, 6 layers, dimension size = 320.
esm2_model, alphabet = esm.pretrained.esm2_t6_8M_UR50D() # 6 layers
esm2_model = esm2_model.to(device)
esm2_batch_converter = alphabet.get_batch_converter()

# EXAMPLE


In [4]:
uvsx_t=read_fasta('../uvsx/data/references/7Z3M.fasta')
uvsx_p=read_fasta('../uvsx/data/references/9GBG.fasta')
uvsx=read_fasta('../uvsx/data/references/P04529.fasta')

In [5]:
def predict(seq):
    # seq: input protein sequence
    inputs = [('Temp', seq)]
    batch_labels, batch_strs, batch_tokens = esm2_batch_converter(inputs)
    batch_tokens = batch_tokens.to(device=device, non_blocking=True)
    with torch.no_grad():
        emb = esm2_model(batch_tokens, repr_layers=[6], return_contacts=False)
    emb = emb["representations"][6]
    emb = emb.transpose(1,2)
    emb = emb.to(device)
    with torch.no_grad():
        preds = model( emb )
    return preds

In [6]:
# The prediction output is Topt/Topt_max.
Topt_max=120
print(f'UvsXt opt = {int(predict(uvsx_t) * Topt_max)}')
print(f'UvsXp opt = {int(predict(uvsx_p) * Topt_max)}')
print(f'UvsX opt = {int(predict(uvsx) * Topt_max)}')

UvsXt opt = 45
UvsXp opt = 52
UvsX opt = 42


# Load TM model

In [7]:
# path to model weights
tm_pth = '../seq2topt/model_tm_window.3_r2.0.76.pth' # you can download the weights from release v1.0.0
# define device
device = torch.device('cpu')
# define seq2Topt model
emb_dim= 320; window=3; n_head = 4; n_RD = 4;
model = MultiAttModel( emb_dim, window, n_head, n_RD)
model.to(device);
# load weights into the model
model.load_state_dict(torch.load( tm_pth, map_location=device  ));
model.eval();

# Load ESM

In [8]:
# We use esm2_t6_8M_UR50D, 6 layers, dimension size = 320.
esm2_model, alphabet = esm.pretrained.esm2_t6_8M_UR50D() # 6 layers
esm2_model = esm2_model.to(device)
esm2_batch_converter = alphabet.get_batch_converter()

In [9]:
# We use esm2_t6_8M_UR50D, 6 layers, dimension size = 320.
esm2_model, alphabet = esm.pretrained.esm2_t6_8M_UR50D() # 6 layers
esm2_model = esm2_model.to(device)
esm2_batch_converter = alphabet.get_batch_converter()

In [10]:
# The prediction output is Topt/Topt_max.
Topt_max=120
print(f'UvsXt opt = {int(predict(uvsx_t) * Topt_max)}')
print(f'UvsXp opt = {int(predict(uvsx_p) * Topt_max)}')
print(f'UvsX opt = {int(predict(uvsx) * Topt_max)}')

UvsXt opt = 66
UvsXp opt = 74
UvsX opt = 65


For both models UvsXp is higher than UvsXt.

Temperatures seem much higher than expected.
From the paper seq2topt as tested using mesophilic, thermophilic and hyperthermophilic microorganisms. The lack of Psychrophilic micoorganisms
may means it performs poorly when trying to predict their optimal temps.


Ambitious but maybe I could try to build a model that predicts optimal tempt from crystal/alphafold structure.
